# Graph Construction and Querying with Cypher

In [ ]:
import getpass

# configuration llm
google_llm_model = "gemma-4-31b-it"
# google_embeder_model = "gemini-embedding-2"
google_api_key = getpass.getpass("Enter your Google AI API key: ")

# configuration embedder
hugging_face_embeder_model = "google/embeddinggemma-300m"
hugging_face_key = getpass.getpass("Enter your Hugging Face API key: ")

# configuration databasis
neo4j_uri="neo4j+s://312151bc.databases.neo4j.io"
neo4j_username="312151bc"
neo4j_password = getpass.getpass("Enter your Neo4j password: ")
neo4j_database="312151bc"

## Check Ontology

In [2]:
from pymprev.ontology import Ontology

ontology = Ontology()
ontology.description

'This ontology reflects relevant aspects in the development, maintenance or deployment ofproducts and components based on or using Artificial Inteligence and Machine Learning,with focus on recomendation compliance and safety, in the context of civil aviation.\n\nThe node schemas (properties and descriptions): \n\tAiml_element:\n\t * description: elements or components. systems and subsystems, models, tools and products based on or using AI/ML: Neural Nets, Models, etc\n\t * properties: definition, purpose, name, description\n\tAiml_resource:\n\t * description: training sets, generators, accelerators (GPU/FPGA), libraries, parameters, hyperparameters\n\t * properties: definition, name, description\n\tAiml_lifecycle_aspect:\n\t * description: Aspects related to AI/ML development and maintenance: implementation, training, validation, deployment, etc\n\t * properties: key_deliverable, definition, name, description\n\tRisk_factor:\n\t * description: Risks system or component may be subject 

## Connection and File Ingestion

In [3]:
from pymprev.connection import connect_llm, connect_embeder, connect_database

/home/fredson-aguiar/Downloads/venv-mprev/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# connect services
llm = connect_llm(
    google_api_key,
    google_llm_model)

embedder = connect_embeder(
    hugging_face_key,
    hugging_face_embeder_model,
    model_kwargs={"device": "cpu"}) # "cuda" 

graph = connect_database(
    neo4j_uri,
    neo4j_database,
    neo4j_username,
    # sanitize=True,
    # enhanced_schema=True,
    neo4j_password)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 11501.68it/s]


In [5]:
from pymprev.util import GraphRAG

graphRAG = GraphRAG(llm, embedder, graph, ontology)

### Documents Ingestion

In [ ]:
graphRAG.ingest_documents("/home/fredson-aguiar/Downloads/resources/")

In [ ]:
graphRAG.unify_nodes_by_similarity(
    high_threshold=0.995, low_threshold=0.95, text_threshold=0.95)

total merged nodes of type Aiml_element: 227
total merged nodes of type Aiml_resource: 72
total merged nodes of type Aiml_lifecycle_aspect: 117
total merged nodes of type Risk_factor: 88
total merged nodes of type Risk_mitigation_strategy: 124


In [9]:
graphRAG.add_knn_similarity_relations(score_threshold=0.95, top_k=5)

finding similar neighbors per type: Aiml_element
finding similar neighbors per type: Aiml_resource
finding similar neighbors per type: Aiml_lifecycle_aspect
finding similar neighbors per type: Risk_factor
finding similar neighbors per type: Risk_mitigation_strategy


## Create Bottom-up Clusterization

In [ ]:
graphRAG.run_leiden_clustering(
    max_cluster_size=15, resolution=0.5, ontology_weight=1.0)

Adding Leiden cluster structure to Database
Creating bottom-up cluster summarization: level 7


100%|██████████| 2/2 [00:07<00:00,  3.73s/it]


Creating bottom-up cluster summarization: level 6


100%|██████████| 13/13 [00:50<00:00,  3.88s/it]


Creating bottom-up cluster summarization: level 5


...

100%|██████████| 313/313 [23:10<00:00,  4.44s/it]


Creating bottom-up cluster summarization: level 1


100%|██████████| 80/80 [06:46<00:00,  5.08s/it]


Creating bottom-up cluster summarization: level 0


100%|██████████| 17/17 [01:22<00:00,  4.83s/it]


In [ ]:
# graphRAG.reset_cluster_embeddings(batch_size=10)

### Evaluate Results

In [ ]:
cluster_ids = graphRAG.get_cluster_ids_by_level(0)
for cluster_id in cluster_ids:
    n_childnodes = len(graphRAG.get_children_entities(cluster_id))
    n_subclusters = len(graphRAG.get_children_clusters(cluster_id))
    name, summary, level = graphRAG.get_cluster_details(cluster_id)

    print(f"cluster {cluster_id}, {n_subclusters+n_childnodes} children, '{name}': {summary}")

cluster 0, 8 children, 'AI Governance, Safety, and Risk Management': This cluster synthesizes a multi-dimensional framework for the deployment of trustworthy AI/ML systems in civil aviation, integrating technical, regulatory, and ethical governance. It encompasses the full AI/ML lifecycle—from foundational infrastructure (HPC, cloud, and data governance) and resource management to the operational integration of human-AI teaming and systemic safety. The framework addresses a broad spectrum of risk factors, including technical vulnerabilities (adversarial attacks, model opacity, and distribution drift), human-centric risks (automation bias and complacency), and socio-environmental impacts (algorithmic bias, human rights violations, and ecological footprints). To ensure safety-critical reliability and regulatory compliance (e.g., AI Act, GDPR), the cluster integrates rigorous risk mitigation strategies, such as eXplainable AI (XAI), formal verification, human-in-the-loop (HITL) oversight,

In [26]:
cluster_id = 7

childnodes = graphRAG.get_children_entities(cluster_id)
subclusters = graphRAG.get_children_clusters(cluster_id)

for r in childnodes:
    text = graphRAG._node_as_text(r['node_id'], r['labels'], r['name'], r['description'])
    print(text)

for r in subclusters:
    text = graphRAG._cluster_as_text(r['child_id'], r['name'], r['summary'])
    print(text)

Cluster ID: 78 
	* name: Speech and Voice Processing Systems
	* description: This cluster focuses on the development and deployment of AI/ML elements specialized in human language interpretation, encompassing Speech-to-Text (STT), Text-to-Speech (TTS), and voice recognition systems. It integrates cognitive systems for acoustic modeling and identity validation (voiceprints) using tools like Speechbrain and Unispeech. Within the context of civil aviation, these technologies are applied to Automatic Speech Recognition (ASR) for transcribing Air Traffic Control (ATC) communications to reduce controller workload, utilizing architectures such as DeepSpeech, Kaldi, and WaveNet. The cluster also addresses critical risk factors, including adversarial attacks and operational challenges like multilingual accents and lexical ambiguities, aligning with the ontology's focus on safety, compliance, and risk mitigation in aviation environments.
Cluster ID: 79 
	* name: ATC Speech-to-Text Systems and Ri

In [ ]:
cluster_id = 1427

childnodes = graphRAG.get_children_entities(cluster_id)
subclusters = graphRAG.get_children_clusters(cluster_id)

for r in childnodes:
    text = graphRAG._node_as_text(r['node_id'], r['labels'], r['name'], r['description'])
    print(text)

for r in subclusters:
    text = graphRAG._cluster_as_text(r['child_id'], r['name'], r['summary'])
    print(text)

Node ID/Type: Validation And Certification Process/['Aiml_lifecycle_aspect'] 
	* name: None
	* description: The process of validating and certifying the SHM system based on application intent and criticality
Node ID/Type: Validation Program/['Aiml_lifecycle_aspect'] 
	* name: None
	* description: Program to establish that the SHM system is effective in detecting damage
Node ID/Type: Shm Validation Processes/['Aiml_lifecycle_aspect'] 
	* name: None
	* description: Processes to validate SHM solutions, including objective evaluation of skills, automation, and human error, and performance assessment


## Query Similarity Retrieval

In [6]:
result = graphRAG.query_RAG(
    "What safety problems are associated to Neural Nets in Aviation?",
    top_k_global=10, top_k_local=10)

In [7]:
print("title:", result["title"])
print("answer:", result["answer"])

title: Safety Problems Associated with Neural Networks in Aviation
answer: Safety problems associated with Neural Networks (NNs) in aviation primarily stem from their inherent complexity and the 'black box' nature of their internal logic, which creates significant challenges for traditional safety assurance. Key issues include:

1. **Verification and Traceability Challenges**: There is a lack of traceability between the weights and interconnects of an ANN and its high-level system requirements, making it nearly impossible to perform reverse traces from the implementation to the requirements. Traditional requirements-based testing and coverage analysis (e.g., DO-331) are often incompatible with the training and validation phases of NNs.

2. **Robustness and Unpredictability**: ML implementations can react in unexpected and incorrect ways to even slight perturbations of inputs (adversarial inputs), which can result in unsafe systems. Furthermore, the training sets used may be incomplete 

In [8]:
result["clusters"]

[{'id': 28,
  'name': 'Neural Network Architecture and Verification',
  'summary': 'This cluster encompasses the design, structural configuration, and formal verification of Artificial Neural Networks (ANNs) and Deep Neural Networks (DNNs) within civil aviation safety-critical systems. It covers a diverse range of architectures—including CNNs, RNNs, LSTMs, and Feedforward Neural Networks (FNNs) specifically implemented in systems like ACAS Xu—alongside fundamental components such as activation functions, loss functions, and layer topologies. To ensure compliance and safety, the cluster integrates rigorous risk mitigation strategies, such as the use of SMT solvers, L-Lipschitz architectures, and neuron coverage analysis to address risks like dead neurons, non-linearity, and inconsistent alerting. The focus remains on utilizing formal specification formats (e.g., NNEF) and verification frameworks (e.g., Dnnv) to provide quantitative feedback and ensure robust behavioral reactions to oper

In [11]:
result["chunks"]

[{'text': '30 / The FLY AI Report 2020\n7. Guarantee the safe use of AI\n7.1. AI CAN IMPROVE SAFETY\nControlling and improving Aviation safety has historically \nbeen achieved using both comprehensive incidents / \naccidents analyses to stop them occurring as well as safety \nassessments and safety cases documenting and recording the \nsafety of aviation services or systems. They both rely on service \nexperience, i.e. data from previous operational use. Proof of \npast safety achievements has turned out to be difficult since \nsuitable baseline did not always exist or sufficient historical \ndata was not available. Hence quite often, safety evidence \nhas been based on expert judgements. Digitalisation and \nAI, however, open up new possibilities for aviation safety, \nas huge amounts of data can now be processed in order to \nidentify unknown incident patterns and early detection of \nprecursors (‘weak signals’), including previously hard-to-\nanalyse data such as informal written re

In [12]:
unique_sources = set()
for chunk in result["chunks"]:
    unique_sources.add(chunk["source"])

unique_sources

{'/home/fredson-aguiar/Downloads/resources/Machine Learning Safety and Applicability/AFE 87 – Machine Learning - Final Report.pdf',
 '/home/fredson-aguiar/Downloads/resources/Machine Learning Safety and Applicability/Eurocontrol-fly-ai-report-032020.pdf',
 '/home/fredson-aguiar/Downloads/resources/Machine Learning Safety and Applicability/IEEE AC (2023) - BNAE - Trusting ML App in Aeronautic (v1.1).pdf'}

### Reseting Databasis, Embeddings or Removing document

In [8]:
# # reset complete databasis
# graphRAG.reset_databasis(graph)

In [ ]:
# # reset embeddings
# graphRAG.reset_chunk_embeddings(10)
# graphRAG.reset_node_embeddings(10)
# graphRAG.reset_cluster_embeddings(10)

In [ ]:
# # delete clusters
# graphRAG.delete_clustering()

In [10]:
# # removing single file and instances created from it
# filename = "/home/fredson-aguiar/Downloads/resources/Machine Learning Safety and Applicability/ARP6983 v8_260306.pdf"
# graphRAG.remove_document(graph, filename=filename)